#### Fetch census data

In [ ]:
# %pip install census
# %pip install python-dotenv

<!-- Census endpoint -->

In [ ]:
year = 2024                     # 2020-2024 ACS 5-year
dataset = "acs/acs5"            # 5-year detailed tables
base = f"https://api.census.gov/data/{year}/{dataset}"

# api_key = None                  # optional; free key at api.census.gov/data/key_signup.html

In [ ]:
# https://api.census.gov/data/2024/acs/acs5/variables.html
variables = {
    "NAME": "name",
    "B01003_001E": "population",
    "B03002_012E": "hispanic_latino",
    'B03002_003E':'non_hispanic_white',
    'B03002_004E':'non_hispanic_black',
    'B03002_005E':'non_hispanic_native_american',
    'B03002_006E':'non_hispanic_asian',
    'B03002_007E':'non_hispanic_pacific_islander',
    'B03002_008E':'non_hispanic_other',
    'B03002_009E':'non_hispanic_multiracial',
    "B19013_001E": "median_hh_income",
}

variables

In [ ]:
fields=list(variables.keys())

fields

In [ ]:
# %pip install us

In [ ]:
import us

state_fips = {s.abbr: s.fips for s in us.states.STATES}
state_fips.keys()

In [ ]:
from census import Census
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())
census_key = os.getenv("census_key")

c = Census(census_key)

print(census_key)

In [ ]:
# state
c.acs5.state(fields, state_fips['CA'], year=2024)

In [ ]:
# counties in a state
c.acs5.state_county(fields, state_fips['CA'], Census.ALL, year=2024)[0]

In [ ]:
# tracts in a state (all counties)
c.acs5.state_county_tract(fields, state_fips['CA'], Census.ALL, Census.ALL, year=2024)[0]


In [ ]:
def fetch_demo(abbr, level="state"):
    fips = state_fips[abbr]
    fields = list(variables.keys())

    if level == "state":
        rows = c.acs5.state(fields, fips, year=2024)
    elif level == "county":
        rows = c.acs5.state_county(fields, fips, Census.ALL, year=2024)
    elif level == "tract":
        rows = c.acs5.state_county_tract(fields, fips, Census.ALL, Census.ALL, year=2024)

    df = pd.DataFrame(rows)
    df["state_abbr"] = abbr
    return df

# test on one state first
fetch_demo("CA", level="county").head()

In [ ]:
dfs=[]
for abbr in state_fips.keys():
    df = fetch_demo(abbr, level="state")
    print(f"Fetched data for {abbr}")
    dfs.append(df)

df=pd.concat(dfs, ignore_index=True)

In [ ]:
df = pd.DataFrame(c.acs5.state(fields, Census.ALL, year=2024))

In [ ]:
fips=list(state_fips.values())

In [ ]:
rows = c.acs5.state_county(fields, "01", Census.ALL, year=2024)
df = pd.DataFrame(rows)

In [ ]:
df.rename(columns=variables, inplace=True)